# COMP90049 A2 — 数据清洗与特征聚合

**OULAD (Open University Learning Analytics Dataset)**
Kuzilek, J., Hlosta, M., & Zdrahal, Z. (2017). *Scientific Data*, 4, 170171.
数据来源：https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset (DOI: 10.24432/C5KK69)

本 notebook 负责**数据层**：清洗 → 聚合 → 产出可直接建模的宽表，以及统一的两套特征集。
建模与评估在后续 notebook / 由队友接手，使用本文件输出的宽表。

---

## ⚠️ 你必须能在报告里讲清楚的四处

跑完之后请重点看这四个地方，它们是 Method 与 Discussion 的核心论证：

| # | 位置 | 要能回答 |
|---|---|---|
| 1 | Cell「证据 1」 | 为什么必须用 `na_values=['?']`？不加会发生什么？ |
| 2 | Cell「证据 2」 | 为什么 `date_unregistration` 不能当特征，但又必须读进来？ |
| 3 | Cell「证据 3」 | 为什么要按 `id_student` 分组划分，而不是随机划分？ |
| 4 | Cell「风险集」 | 为什么在第 t 天预测时要排除「已经退课」的学生？ |

---

## 本次已锁定的决策

| 项 | 决定 | 依据 |
|---|---|---|
| 目标变量 | **方案 A**：未完成课程 (Fail + Withdrawn) = 正类 1；完成 (Pass + Distinction) = 0 | 52.79% / 47.21%，接近平衡；且学校关心的是「找出高风险学生」 |
| 样本粒度 | `(id_student, code_module, code_presentation)` 三元组 | 已验证三元组唯一 |
| 数据规模 | 全量处理 1,065 万行点击流 | 聚合后仅约 3 万行，无需抽样，少一个要辩护的决定 |
| 划分方式 | 按 `id_student` 分组（本 notebook 只产出 group 列，划分在建模阶段做） | 3,538 名学生跨多条记录 |
| 必删列 | `date_unregistration`（目标泄漏）、`week_from`/`week_to`（82.39% 缺失） | 见证据 2 |
| 截断点 | 0 / 24 / 60 / 120 天，锚定课程中的真实事件 | 首次作业提交 75% 分位 = 24 天 |


## 0. 配置

In [ ]:
from pathlib import Path

# ====== 路径（相对仓库根目录，照 README 的约定）======
DATA_DIR = Path("data")
OUT_DIR  = Path("processed")
# 七个 CSV 直接放在 data/ 下，见 README
# ==============================================

OUT_DIR.mkdir(parents=True, exist_ok=True)

# 全局随机种子：保证每次跑出来的数字完全一致
# （去年有同学没设，报告与代码结果不符被扣分）
SEED = 42

# 样本主键
KEY = ["code_module", "code_presentation", "id_student"]

# 截断点（天，相对开课日 date=0）。每个都对应课程中的一个真实节点：
#   0   开课日        —— 课程尚未开始，只有背景特征可用
#   24  第一次作业已交 —— 首次提交时间的 75% 分位数 = 第 24 天
#   60  第二次作业已交
#   120 课程中点      —— 课程长度 234–269 天
CUTOFF_DAYS = [0, 24, 60, 120]

# studentVle 中最早的 date（有开课前活动），用于计算窗口长度
VLE_WINDOW_START = -25

print("DATA_DIR:", DATA_DIR, "存在" if DATA_DIR.exists() else "❌ 不存在，请检查路径")
print("OUT_DIR :", OUT_DIR)

## 1. 读取数据

⚠️ **关键**：OULAD 用字符串 `?` 表示缺失。不声明 `na_values=['?']` 的话：
- 缺失率会被错误地报告为 0%
- `date_registration`、`score` 等本该是数值的列会被读成字符串，无法计算、无法建模

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

NA = ["?"]          # ← 这一行是整个 notebook 最重要的参数之一

student_info = pd.read_csv(DATA_DIR / "studentInfo.csv",         na_values=NA)
student_reg  = pd.read_csv(DATA_DIR / "studentRegistration.csv", na_values=NA)
student_asm  = pd.read_csv(DATA_DIR / "studentAssessment.csv",   na_values=NA)
assessments  = pd.read_csv(DATA_DIR / "assessments.csv",         na_values=NA)
vle_meta     = pd.read_csv(DATA_DIR / "vle.csv",                 na_values=NA)
courses      = pd.read_csv(DATA_DIR / "courses.csv",             na_values=NA)

for name, df in [("studentInfo", student_info), ("studentRegistration", student_reg),
                 ("studentAssessment", student_asm), ("assessments", assessments),
                 ("vle", vle_meta), ("courses", courses)]:
    print(f"{name:22s} {str(df.shape):>16s}")

In [ ]:
# studentVle 有 1,065 万行 / 453 MB。指定 dtype 可把内存从约 1.5 GB 降到约 250 MB。
# date 范围 -25..269，装得进 int16。
VLE_DTYPES = {
    "code_module":       "category",
    "code_presentation": "category",
    "id_student":        "int32",
    "id_site":           "int32",
    "date":              "int16",
    "sum_click":         "int32",
}

student_vle = pd.read_csv(DATA_DIR / "studentVle.csv", dtype=VLE_DTYPES, na_values=NA)
print("studentVle           ", student_vle.shape)
print("内存占用: %.0f MB" % (student_vle.memory_usage(deep=True).sum() / 1e6))
print("\ndate 范围: %d .. %d" % (student_vle["date"].min(), student_vle["date"].max()))

## 2. 证据 1：`?` 伪缺失值

这一节产出报告 Method 3.1 要引用的数字。**不要删掉，报告里的数字必须能在代码里找到出处。**

In [ ]:
evidence_1 = []
for name, path in [("studentInfo", "studentInfo.csv"),
                   ("studentRegistration", "studentRegistration.csv"),
                   ("studentAssessment", "studentAssessment.csv"),
                   ("vle", "vle.csv")]:
    raw = pd.read_csv(DATA_DIR / path)            # 故意不加 na_values，还原「默认读取」的样子
    fixed = pd.read_csv(DATA_DIR / path, na_values=NA)
    for col in raw.columns:
        n = (raw[col].astype(str).str.strip() == "?").sum()
        if n:
            evidence_1.append({
                "表": name, "列": col,
                "'?' 数量": n,
                "占比%": round(n / len(raw) * 100, 2),
                "默认读取的 dtype": str(raw[col].dtype),
                "修正后的 dtype": str(fixed[col].dtype),
            })

evidence_1 = pd.DataFrame(evidence_1)
print("证据 1：以 '?' 编码的缺失值")
display(evidence_1)
evidence_1.to_csv(OUT_DIR / "evidence_1_hidden_missing.csv", index=False)

print("\n→ 报告写法：该数据集以字符串 '?' 而非空值表示缺失。若沿用默认读取设置，")
print("  date_registration 与 score 等数值字段会被解析为字符串，缺失率亦被错误报告为 0%。")

## 3. 证据 2：`date_unregistration` 是目标泄漏

这一列**必须读进来**（后面定义风险集要用），但**绝不能进特征**。

In [ ]:
chk = student_info.merge(student_reg, on=KEY, how="left")
chk["无退课日期"] = chk["date_unregistration"].isna()

evidence_2 = pd.crosstab(chk["final_result"], chk["无退课日期"], normalize="index").round(4) * 100
evidence_2.columns = ["有退课日期 %", "无退课日期 %"]
print("证据 2：date_unregistration 的缺失情况 vs final_result")
display(evidence_2)
evidence_2.to_csv(OUT_DIR / "evidence_2_leakage.csv")

print("\n→ 该列有值 ⟺ 学生退课，准确率 >99%。若保留为特征，ROC-AUC 会接近 1.0，结果无意义。")
print("→ 但我们仍需读取它来判断『某学生在第 t 天是否已退课』（见风险集一节）。")
print("→ 69.1% 的缺失率不是数据质量缺陷，而是该字段的结构性特征。")

## 4. 证据 3：分组结构（为什么不能随机划分）

In [ ]:
vc = student_info["id_student"].value_counts()
n_multi = int((vc > 1).sum())

print("证据 3：分组结构")
print(f"  总记录数                     = {len(student_info):,}")
print(f"  唯一 id_student              = {student_info['id_student'].nunique():,}")
print(f"  唯一 (学生,课程,学期) 三元组 = {student_info.groupby(KEY).ngroups:,}  ← 与总记录数相等，三元组唯一")
print(f"  出现于多条记录的学生         = {n_multi:,}（最多 {vc.max()} 条）")

# 同一学生的多次结果是否相关？—— 这是分组划分的实证依据
si = student_info.copy()
si["completed"] = si["final_result"].isin(["Pass", "Distinction"])
_order = {"2013B": 0, "2013J": 1, "2014B": 2, "2014J": 3}
multi = (si[si["id_student"].isin(vc[vc > 1].index)]
         .assign(_o=lambda d: d["code_presentation"].map(_order))
         .sort_values(["id_student", "_o"]))
pairs = multi.groupby("id_student")["completed"].apply(list)
pairs = pairs[pairs.apply(len) >= 2]
first = pairs.apply(lambda x: x[0])
second = pairs.apply(lambda x: x[1])

print(f"\n  两次结果一致率              = {(first == second).mean()*100:.1f}%")
print(f"  第一次完成 → 第二次完成率   = {second[first].mean()*100:.1f}%")
print(f"  第一次未完成 → 第二次完成率 = {second[~first].mean()*100:.1f}%")
print(f"  全体完成率（基准）          = {si['completed'].mean()*100:.1f}%")
print("\n→ 条件完成率相差约 31 个百分点：同一学生的记录高度相关，不是独立样本。")
print("→ 若随机划分，模型可凭『认出这个人』在测试集作弊 → 分数虚高。")
print("→ 因此必须按 id_student 分组划分（GroupShuffleSplit / StratifiedGroupKFold）。")

## 5. 清洗 studentInfo + 构造目标变量

**目标变量（方案 A）**：`at_risk = 1` 表示未完成课程（Fail 或 Withdrawn）。

把「未完成」设为正类，是为了让 precision / recall 直接对应业务含义：
- **precision** = 预警了的学生里，真的有风险的比例（「别乱报警」）
- **recall** = 真有风险的学生里，被预警到的比例（「别漏掉」）

In [ ]:
info = student_info.copy()

# ---- 目标变量（方案 A）----
info["at_risk"] = (~info["final_result"].isin(["Pass", "Distinction"])).astype("int8")

print("目标变量分布：")
print(info["at_risk"].value_counts().rename({0: "0 已完成", 1: "1 未完成"}).to_string())
print((info["at_risk"].value_counts(normalize=True) * 100).round(2).to_string())
print("\n原始四分类：")
print(info["final_result"].value_counts().to_string())

In [ ]:
# ---- imd_band：修掉格式 bug ----
# 原始取值中 '10-20' 缺少百分号，与其余九个分段格式不一致（3,516 条）。
# 不修的话 one-hot 会把它当成独立类别。
print("修正前:", sorted(info["imd_band"].dropna().unique().tolist()))

info["imd_band"] = (info["imd_band"].astype("string").str.strip()
                    .str.replace(r"(?<!%)$", "%", regex=True))
print("修正后:", sorted(info["imd_band"].dropna().unique().tolist()))

# imd_band 与 highest_education 都是**有序**类别 (ordinal)。
# 用 one-hot 会丢掉顺序信息，所以转成数值 —— 这是一个有论证的非默认决定。
IMD_MAP = {f"{i}-{i+10}%": i + 5 for i in range(0, 100, 10)}
IMD_MAP["0-10%"] = 5
info["imd_band_num"] = info["imd_band"].map(IMD_MAP).astype("float32")

EDU_MAP = {"No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
           "HE Qualification": 3, "Post Graduate Qualification": 4}
info["highest_education_ord"] = info["highest_education"].map(EDU_MAP).astype("float32")

AGE_MAP = {"0-35": 0, "35-55": 1, "55<=": 2}
info["age_band_ord"] = info["age_band"].map(AGE_MAP).astype("float32")

# 核查映射是否有漏（除 imd_band 本身的 3.41% 缺失外，不该有新的 NaN）
print("\n映射后缺失率 %:")
print(info[["imd_band_num", "highest_education_ord", "age_band_ord"]].isna().mean().mul(100).round(2).to_string())

In [ ]:
# ---- 合并注册信息 ----
# date_registration：负数 = 开课前多少天报名（报名提前期），是合法特征
# date_unregistration：保留在表中仅用于风险集判断，**不进特征**
reg = student_reg.copy()
info = info.merge(reg, on=KEY, how="left", validate="one_to_one")

info["reg_lead_days"] = -info["date_registration"]      # 转成正数更好读：越大 = 越早报名
info["reg_date_missing"] = info["date_registration"].isna().astype("int8")

# 开课之后才报名的记录：234 条（0.72%），最晚的一条在课程 70% 处。
# 数量极少、不影响结论，但这是一项真实的数据质量异常，因此保留并加标记而非删除。
info["registered_after_start"] = (info["date_registration"] > 0).fillna(False).astype("int8")
print("\n开课后才报名: %d 条 (%.2f%%)，最晚 %d 天" % (
    info["registered_after_start"].sum(),
    info["registered_after_start"].mean()*100,
    info["date_registration"].max()))

# ---- 合并课程长度 ----
# 课程长度 234–269 天不等，所以「活跃天数」这个绝对值不可比，
# 后面要除以课程长度转成比例 —— 又一个有论证的决定。
info = info.merge(courses, on=["code_module", "code_presentation"], how="left")

print("合并后 shape:", info.shape)
print("module_presentation_length 范围: %d .. %d" % (
    info["module_presentation_length"].min(), info["module_presentation_length"].max()))
assert len(info) == len(student_info), "合并后行数变了，检查 join 键"
print("✅ 行数未变")

## 6. 准备 VLE 与作业数据

In [ ]:
# id_site → activity_type 的映射（vle.csv 里 id_site 唯一）
assert vle_meta["id_site"].is_unique
site2type = vle_meta.set_index("id_site")["activity_type"]

# 固定全部 activity_type 的清单。
# 必须这么做：像 repeatactivity 这类资源要到开课一段时间后才有人点击，
# 若按各截断点实际出现的类型动态生成列，早期截断点会少几列，
# 导致各时点的特征集不一致、RQ1 的跨时点对比失效。
ALL_ACTIVITY_TYPES = sorted(vle_meta["activity_type"].dropna().unique().tolist())
print("固定 activity_type 列数 =", len(ALL_ACTIVITY_TYPES))
print("activity_type 共 %d 种:" % vle_meta["activity_type"].nunique())
print(sorted(vle_meta["activity_type"].unique().tolist()))

# week_from / week_to 缺失 82.39%，基本不可用 → 不使用
print("\nweek_from / week_to 缺失率 %:",
      vle_meta[["week_from", "week_to"]].isna().mean().mul(100).round(2).to_dict())

student_vle["activity_type"] = student_vle["id_site"].map(site2type).astype("category")
print("\n映射后 activity_type 缺失数:", int(student_vle["activity_type"].isna().sum()))

In [ ]:
# studentAssessment 没有 code_module / code_presentation，必须经 assessments 表 join 进来。
# assessments 还提供：date（截止日）、weight（权重）、assessment_type
asm = student_asm.merge(
    assessments[["id_assessment", "code_module", "code_presentation",
                 "assessment_type", "date", "weight"]],
    on="id_assessment", how="left", validate="many_to_one")

asm = asm.rename(columns={"date": "due_date"})
asm["days_early"] = asm["due_date"] - asm["date_submitted"]   # 正 = 提前交，负 = 迟交

print("作业表 shape:", asm.shape)
print("缺失率 %:", asm.isna().mean().mul(100).round(2).to_dict())
print("\ndays_early 分布:")
print(asm["days_early"].describe().round(1).to_string())

## 7. 特征聚合函数

核心逻辑：给定截断点 `cutoff`，只用 `date <= cutoff` 的记录构造特征。

**为什么要截断**：退课学生退课后不再产生点击记录。若用全程数据，「活跃天数少」与「退课」会形成**反向因果**——是退课导致无点击，而非无点击导致退课。

In [ ]:
def _normalise_key(f):
    """把聚合结果的主键 dtype 统一成 str。
    studentVle 为省内存用了 category dtype，若不统一，
    MultiIndex join 时 CategoricalIndex 与 object index 会对不上，结果全是 NaN。"""
    f = f.reset_index()
    f["code_module"] = f["code_module"].astype(str)
    f["code_presentation"] = f["code_presentation"].astype(str)
    f["id_student"] = f["id_student"].astype("int64")
    return f.set_index(KEY)


def vle_features(svle, cutoff):
    """把点击流聚合成『一个学生一行』。只用 date <= cutoff 的记录。"""
    w = svle[svle["date"] <= cutoff]
    if len(w) == 0:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=KEY))

    g = w.groupby(KEY, observed=True)
    f = g.agg(
        vle_total_clicks=("sum_click", "sum"),
        vle_active_days=("date", "nunique"),
        vle_n_sites=("id_site", "nunique"),
        vle_max_clicks_day=("sum_click", "max"),
    )
    f["vle_mean_clicks_per_active_day"] = f["vle_total_clicks"] / f["vle_active_days"]

    # 趋势特征：窗口后半段 vs 前半段的点击量。
    # 直觉：活动量「下降」比「绝对量低」更能预示放弃。
    mid = (VLE_WINDOW_START + cutoff) / 2
    h1 = w[w["date"] <= mid].groupby(KEY, observed=True)["sum_click"].sum()
    h2 = w[w["date"] >  mid].groupby(KEY, observed=True)["sum_click"].sum()
    f["vle_clicks_h1"] = h1.reindex(f.index).fillna(0)
    f["vle_clicks_h2"] = h2.reindex(f.index).fillna(0)
    f["vle_trend_ratio"] = (f["vle_clicks_h2"] + 1) / (f["vle_clicks_h1"] + 1)   # +1 避免除零

    # 开课前的预习活动（只在 cutoff > 0 时有意义）
    pre = w[w["date"] <= 0].groupby(KEY, observed=True)["sum_click"].sum()
    f["vle_clicks_before_start"] = pre.reindex(f.index).fillna(0)

    # 各类资源的点击占比（结构特征，而非总量）
    at = (w.groupby(KEY + ["activity_type"], observed=True)["sum_click"]
            .sum().unstack(fill_value=0))
    at = at.div(at.sum(axis=1).replace(0, np.nan), axis=0)
    # 补齐到完整的 activity_type 清单，保证各截断点列集合一致
    at = at.reindex(columns=ALL_ACTIVITY_TYPES, fill_value=0.0)
    at.columns = ["vle_prop_" + str(col) for col in at.columns]
    f = f.join(at)

    return _normalise_key(f)


def asm_features(asm_df, cutoff):
    """把作业提交记录聚合成『一个学生一行』。只用 date_submitted <= cutoff 的记录。"""
    w = asm_df[asm_df["date_submitted"] <= cutoff].copy()
    if len(w) == 0:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=KEY))

    w["_wscore"] = w["score"] * w["weight"]
    g = w.groupby(KEY, observed=True)
    f = g.agg(
        asm_n_submitted=("id_assessment", "nunique"),
        asm_mean_score=("score", "mean"),
        asm_min_score=("score", "min"),
        asm_n_banked=("is_banked", "sum"),
        asm_mean_days_early=("days_early", "mean"),
        _wsum=("_wscore", "sum"),
        _wtot=("weight", "sum"),
    )
    f["asm_weighted_score"] = f["_wsum"] / f["_wtot"].replace(0, np.nan)
    return _normalise_key(f.drop(columns=["_wsum", "_wtot"]))

print("✅ 聚合函数已定义")

## 8. 风险集 (risk set) —— ⚠️ 第 4 处必须能讲清楚的地方

在第 t 天做预测时，如果把「在 t 之前**已经退课**」的学生也当成预测对象，那不是**预测**，而是**记录已经发生的事**。这是一种泄漏。

正确做法：第 t 天只对「截至第 t 天仍在读」的学生建模。这个概念在生存分析 (survival analysis) 中叫**风险集**。

代价：随 t 增大，样本量缩小、类别比例变化。所以**每个时点都要分别报告样本数与类别比例**。

参考：第 0 天前已退课 30.7%，第 24 天约 48%，第 120 天约 73%。

In [ ]:
def build_wide_table(cutoff, verbose=True):
    """产出某个截断点下的完整宽表。"""
    base = info.copy()

    # ---- 风险集过滤：排除在 cutoff 当天或之前已退课的学生 ----
    already_withdrawn = base["date_unregistration"].notna() & (base["date_unregistration"] <= cutoff)
    n_excluded = int(already_withdrawn.sum())
    base = base.loc[~already_withdrawn].copy()

    # ---- 聚合特征 ----
    vf = vle_features(student_vle, cutoff)
    af = asm_features(asm, cutoff)

    # ---- left join：以学生表为准 ----
    # 关键：完全没有点击记录的学生在 vf 里不存在，join 后是 NaN。
    # 这些人**不能丢**——「零交互」本身是最强的预测信号之一，填 0 才是正确语义。
    out = (base.set_index(KEY)
               .join(vf, how="left")
               .join(af, how="left")
               .reset_index())

    vle_cols = [c for c in out.columns if c.startswith("vle_")]
    asm_cols = [c for c in out.columns if c.startswith("asm_")]

    # 用索引成员判断，不依赖列顺序（比看某一列是否 NaN 稳）
    idx = pd.MultiIndex.from_frame(out[KEY])
    out["has_vle_activity"] = idx.isin(vf.index).astype("int8")
    out["has_asm_activity"] = idx.isin(af.index).astype("int8")
    n_zero_vle = int((out["has_vle_activity"] == 0).sum())

    # 计数型特征填 0（真实语义：没有活动）
    count_like = [c for c in vle_cols if "prop" not in c and "trend" not in c and "mean" not in c]
    out[count_like] = out[count_like].fillna(0)
    out[["asm_n_submitted", "asm_n_banked"]] = out[["asm_n_submitted", "asm_n_banked"]].fillna(0)
    # 占比与均值型保留 NaN，交给建模阶段的 imputer 处理（避免用 0 冒充「平均分 0」）

    # ---- 把活跃天数转成比例（课程长度 234–269 天不等，绝对值不可比）----
    window_len = cutoff - VLE_WINDOW_START + 1
    out["vle_active_days_ratio"] = out["vle_active_days"] / window_len

    out["cutoff_day"] = cutoff

    if verbose:
        print(f"\n{'='*64}\ncutoff = 第 {cutoff} 天")
        print(f"{'='*64}")
        print(f"  风险集排除（已退课）     = {n_excluded:,}  ({n_excluded/len(info)*100:.1f}%)")
        print(f"  剩余样本数               = {len(out):,}")
        print(f"  唯一学生数               = {out['id_student'].nunique():,}")
        print(f"  零点击记录               = {n_zero_vle:,}  ({n_zero_vle/len(out)*100:.1f}%)")
        print(f"  正类 (at_risk=1) 占比    = {out['at_risk'].mean()*100:.2f}%")
        print(f"  特征列数                 = {out.shape[1]}")
    return out, dict(cutoff=cutoff, n_excluded=n_excluded, n=len(out),
                     n_students=out["id_student"].nunique(),
                     zero_vle=n_zero_vle, pos_rate=round(out["at_risk"].mean()*100, 2))

print("✅ build_wide_table 已定义")

## 9. 跑所有截断点

In [ ]:
%%time
tables, summary = {}, []
for cut in CUTOFF_DAYS:
    t, s = build_wide_table(cut)
    tables[cut] = t
    summary.append(s)

summary = pd.DataFrame(summary)
print("\n\n各截断点汇总（这张表要进报告 Method 3.3）：")
display(summary)
summary.to_csv(OUT_DIR / "cutoff_summary.csv", index=False)

## 10. 定义两套特征集

这是回答 RQ2（行为数据 vs 背景数据）的基础：两套特征集跑同一批模型，对比差距。

In [ ]:
sample = tables[CUTOFF_DAYS[1]]       # 用第 24 天那张表来确定列名

# ---- 背景特征（报名时就知道的）----
BACKGROUND_CAT = ["code_module", "code_presentation", "gender", "region", "disability"]
BACKGROUND_NUM = ["highest_education_ord", "imd_band_num", "age_band_ord",
                  "num_of_prev_attempts", "studied_credits",
                  "reg_lead_days", "reg_date_missing", "registered_after_start",
                  "module_presentation_length"]

# ---- 行为特征（上课过程中产生的）----
BEHAVIOUR_NUM = [c for c in sample.columns if c.startswith(("vle_", "asm_"))] + \
                ["has_vle_activity", "has_asm_activity"]

# ---- 绝不能进特征的列 ----
FORBIDDEN = ["final_result",            # 目标变量的原始形式
             "at_risk",                 # 目标变量
             "date_unregistration",     # 目标泄漏（证据 2）
             "date_registration",       # 已转为 reg_lead_days
             "imd_band", "highest_education", "age_band",   # 已转为有序数值
             "id_student",              # 身份标识，只作分组键
             "cutoff_day"]

FEATURE_SETS = {
    "baseline":   BACKGROUND_CAT + BACKGROUND_NUM,                  # 只有背景（14）
    "behaviour":  BEHAVIOUR_NUM,                                    # 只有行为（38）
    "engineered": BACKGROUND_CAT + BACKGROUND_NUM + BEHAVIOUR_NUM,  # 背景 + 行为（52）
}

# behaviour 的定义就是 engineered 减 baseline —— 这里断言一次，
# 以后任何人改了上面三个列表，对不上会立刻报错，不会悄悄传错数据给队友。
assert set(FEATURE_SETS["engineered"]) == set(FEATURE_SETS["baseline"]) | set(FEATURE_SETS["behaviour"])
assert not (set(FEATURE_SETS["baseline"]) & set(FEATURE_SETS["behaviour"])), "背景与行为不应有重叠列"

for name, cols in FEATURE_SETS.items():
    missing = [c for c in cols if c not in sample.columns]
    assert not missing, f"{name} 里有不存在的列: {missing}"
    leaked = [c for c in cols if c in FORBIDDEN]
    assert not leaked, f"❌ {name} 里混进了禁用列: {leaked}"
    print(f"{name:12s} {len(cols):3d} 个特征")

print(f"\n其中行为特征 {len(BEHAVIOUR_NUM)} 个:")
print(BEHAVIOUR_NUM)

## 10b. 偏态诊断 → SKEWED_COLS

部分特征呈严重右偏。这会**只影响一部分模型**：

| 模型族 | 是否受影响 | 原因 |
|---|---|---|
| Decision Tree / Random Forest | ❌ 不受影响 | 只做「x > 阈值」比较，仅用序关系。log 是单调变换，不改变任何分裂点 |
| Logistic Regression | ✅ 受影响 | 算加权和 w·x，少数极端值会主导系数 |
| MLP | ✅ 影响最大 | 大输入导致激活饱和或梯度不稳定 |
| KNN / SVM | ✅ 受影响 | 基于距离，大尺度特征会支配距离计算 |

⚠️ **判据不是「偏度 > 2 就取 log」**。`vle_prop_*` 这类 0–1 占比列偏度同样很高（最高 63.7），
但那是**稀疏**（多数学生该类资源点击为 0）而非长尾，log 对其几乎无效（63.7 → 63.1）。

因此我们用**实际效果**作判据：log 后偏度降幅 ≥50%，且是量级型（最大值 > 10）而非占比。

**不在此处变换**，只记录列名。变换作为建模阶段 Pipeline 里的一个开关，
这样同一张宽表可以跑「变换 / 不变换」两个版本，产出 2×4 对照表。

In [ ]:
_num = [c for c in FEATURE_SETS["engineered"] if c not in BACKGROUND_CAT]
_rows = []
for col in _num:
    s = sample[col].dropna()
    if s.nunique() <= 2 or s.min() < 0:
        continue
    sk, skl = s.skew(), np.log1p(s).skew()
    _rows.append({"column": col, "skew": round(sk, 2), "skew_after_log": round(skl, 2),
                  "reduction_%": round((abs(sk) - abs(skl)) / abs(sk) * 100, 1) if sk else 0.0,
                  "max": round(s.max(), 2)})

skew_table = pd.DataFrame(_rows).sort_values("skew", ascending=False)

# 判据：原本重偏 + log 真能把偏度砍掉一半以上 + 量级型而非 0-1 占比
_sel = skew_table[(skew_table["skew"] > 2) &
                  (skew_table["reduction_%"] >= 50) &
                  (skew_table["max"] > 10)]
SKEWED_COLS = _sel["column"].tolist()

print("✅ 选中（log 有效）：")
display(_sel.sort_values("reduction_%", ascending=False))

print("\n❌ 排除（偏度高但 log 无效，多为 0–1 占比列）：")
display(skew_table[(skew_table["skew"] > 2) & (~skew_table["column"].isin(SKEWED_COLS))].head(8))

skew_table.to_csv(OUT_DIR / "evidence_3_skewness.csv", index=False)
print("\nSKEWED_COLS =", SKEWED_COLS)

## 11. 交付前的自检

**这一节是你交给队友之前的验收。全部通过才能推出去。**

In [ ]:
ok = True
def check(label, cond, detail=""):
    global ok
    ok = ok and bool(cond)
    print(("✅ " if cond else "❌ ") + label + ("   " + detail if detail else ""))

t = tables[24]

check("三元组唯一", t.groupby(KEY).ngroups == len(t))
check("目标变量无缺失", t["at_risk"].isna().sum() == 0)
check("目标变量只有 0/1", set(t["at_risk"].unique()) <= {0, 1})
check("禁用列未进入特征集",
      all(c not in FEATURE_SETS["engineered"] for c in FORBIDDEN))
check("date_unregistration 仍在表中（风险集需要）", "date_unregistration" in t.columns)
check("零活动学生被保留",
      (t["has_vle_activity"] == 0).sum() > 0,
      f"零活动 {int((t['has_vle_activity']==0).sum()):,} 条")
check("所有截断点都产出了表", len(tables) == len(CUTOFF_DAYS))
check("随机种子已设定", SEED == 42)

# 诊断：作业特征的缺失率。若 >60%，很可能是 studentAssessment → assessments 的 join
# 没对上（studentAssessment 本身没有 code_module/code_presentation，全靠 id_assessment 带进来）。
_asm_miss = t["asm_mean_score"].isna().mean() * 100
check(f"作业特征缺失率合理（{_asm_miss:.1f}%）", _asm_miss < 60,
      "" if _asm_miss < 60 else "⚠️ 偏高，检查 assessments.csv 的 join 是否正确")

print("\n各截断点的正类占比（应随 cutoff 增大而变化）:")
print(summary[["cutoff", "n", "pos_rate"]].to_string(index=False))

print("\n缺失率最高的 10 个特征（建模阶段需要 imputer 处理）:")
m = t[FEATURE_SETS["engineered"]].isna().mean().mul(100).round(2).sort_values(ascending=False)
print(m.head(10).to_string())

print("\n" + ("="*50))
print("全部自检通过 ✅ 可以交付" if ok else "有自检未通过 ❌ 先修掉再交付")
print("="*50)

## 12. 保存输出

In [ ]:
import json

for cut, t in tables.items():
    path = OUT_DIR / f"wide_cutoff_{cut:03d}.csv"
    t.to_csv(path, index=False)
    print(f"已保存 {path.name:28s} {t.shape}")

manifest = {
    "random_seed": SEED,
    "key": KEY,
    "group_column": "id_student",
    "target": "at_risk",
    "target_definition": "1 = Fail or Withdrawn (未完成); 0 = Pass or Distinction (已完成)",
    "cutoffs": CUTOFF_DAYS,
    "feature_sets": FEATURE_SETS,
    "categorical_features": BACKGROUND_CAT,
    "forbidden_columns": FORBIDDEN,
    "skewed_cols": SKEWED_COLS,
    "notes": [
        "读取时必须使用 na_values=['?']",
        "date_unregistration 保留在表中仅用于风险集判断，禁止作为特征",
        "划分数据集必须按 id_student 分组（GroupShuffleSplit / StratifiedGroupKFold）",
        "预处理必须放进 Pipeline，只对训练集 fit",
        "skewed_cols 列建议在 Pipeline 中套 FunctionTransformer(np.log1p)，并对比变换前后",
    ],
}
with open(OUT_DIR / "manifest.json", "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)
print("\n已保存 manifest.json —— 队友建模前先读这个")

print("\n给队友的一句话：")
print("  读 wide_cutoff_024.csv，目标列是 at_risk，分组列是 id_student，")
print("  特征集见 manifest.json 的 feature_sets，划分必须按 id_student 分组。")

###### ---

## 下一步（建模阶段，不在本 notebook）

1. 用 `StratifiedGroupKFold(groups=id_student)` 或 `GroupShuffleSplit` 划分
2. 预处理放进 `Pipeline` + `ColumnTransformer`，**只对训练集 fit**
   - 数值列：median 填补 + 标准化
   - 类别列：most_frequent 填补 + one-hot
3. `DummyClassifier(strategy="most_frequent")` 跑 majority-class baseline
4. Logistic Regression 验证流水线可用
5. 交给队友：Decision Tree / Random Forest / MLP

## 待办

- [ ] `sum_click` 离群处理（最大 6,977 vs 均值 3.72）：取对数还是截尾
- [ ] `date_unregistration` 异常值（min −365、max 444，课程长度仅 234–269）
- [ ] 同一课程内部比较重修生与首修生（固定课程效应，工作量小、加分明显）
- [ ] 敏感性分析：只看开课后退课者，检验结论稳健性
